#### Imports

In [ ]:
import json
import copy
import os
from dataclasses import asdict
from pathlib import Path
from typing import List, Callable

from collections import defaultdict

from objectives import (
    evaluate_all_objectives,
)
from preprocess import (
    PatientGroundTruth,
    match_ground_truth_and_assistant,
)
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

from dotenv import load_dotenv
from openai import OpenAI
from fastapi.encoders import jsonable_encoder

from dataset.mimic_dataset import MIMIC_Dataset

# Configure logging
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ignore warnings
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

import sys

sys.path.append("..")
from config import DATASET_NAMES
from paths import EVALUABLE_OUTPUTS_HUMAN_BC_DIR, EVALUATIONS_DIR, RAW_DIR

RUN_PARALLEL = False

DIAGNOSIS_MAP = {
    "appendicitis": "appendicitis",
    "cholecystitis": "cholecystitis",
    "diverticulitis": "diverticulitis",
    "pancreatic_cancer": "pancreatic cancer",
    "pancreatitis": "pancreatitis",
    "pneumonia": "pneumonia",
    "pulmonary_embolism": "pulmonary embolism",
    "uti": "urinary tract infection",
}

In [ ]:
DEFAULT_INPUT_DIR = str(EVALUABLE_OUTPUTS_HUMAN_BC_DIR)
DEFAULT_OUTPUT_DIR = str(EVALUATIONS_DIR / "HUMAN_EVALUATED_OUTPUTS")
RUN_PARALLEL = True


#### Load and evaluate human performance

Note: The output format from the humans is slightly different than from MIRA. Therefore we have different evaluation pipelines.

In [ ]:
def get_file_paths_json(base_path: str) -> List[Path]:
    """
    Recursively get all .json file paths under the given base path.

    Args:
        base_path (str): The base directory to search for files.

    Returns:
        List[Path]: A list of file paths.
    """
    print(base_path)
    return list(Path(base_path).rglob("*.json"))


def read_single_json(file_path):
    """
    Reads a single json file and returns the first (and only) entry.
    """
    with open(file_path, "r") as file:
        return json.load(file)  # should be single case only


def load_one_result_human(outputs_path, dataset: MIMIC_Dataset):
    """
    Loads a single result from a jsonl file and returns the medical assistant outputs and patient data.
    """
    evaluable_outputs = read_single_json(outputs_path)
    med_assistant_outputs = evaluable_outputs["med_assistant"]
    dataset_idx = evaluable_outputs["metadata"]["hadm_id"]

    patient_data = dataset[dataset_idx]

    assert (
        patient_data.hadm_id == evaluable_outputs["metadata"]["hadm_id"]
    ), "Patient data and evaluable outputs do not match"

    return med_assistant_outputs, patient_data, evaluable_outputs["metadata"]


def evaluate_one_file(
    client: OpenAI,
    file_path: Path,
    dataset: MIMIC_Dataset,
    out_dir: str,
    diagnosis: str,
    diagnosis_name_for_evaluation: str,
    evaluate_function: Callable[[OpenAI, dict, str], dict],
) -> None:
    """
    Evaluate a single assistant output file against the ground truth and write the evaluation to the output directory.

    Args:
        client (OpenAI): The OpenAI client.
        file_path (Path): The path to the assistant output file.
        dataset (MIMIC_Dataset): The dataset object containing patient data.
        out_dir (str): The base output directory.
        diagnosis (str): The diagnosis for the path creation.
        diagnosis_name_for_evaluation (str): The diagnosis_name_for_evaluation subdirectory name. For use as evaluation criterion in the evaluation.
        evaluate_function (Callable[[OpenAI, dict, str], dict]): The function to evaluate the diagnosis.
    """
    try:
        output_dir = Path(out_dir) / diagnosis
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file_path = output_dir / f"{Path(file_path).stem}.json"

        if os.path.exists(output_file_path):
            logger.info(f"Skipping file {file_path} because it already exists")
            return

        med_assistant_outputs, patient_data, metadata = load_one_result_human(
            file_path, dataset
        )
        print(med_assistant_outputs)

        # Process ground truth and assistant outputs
        patient_gt = PatientGroundTruth(patient_data)
        med_assistant_tools_used = extract_tool_use_human(med_assistant_outputs)
        med_called_args = merge_called_args(med_assistant_tools_used)
        matched_results = match_ground_truth_and_assistant(med_called_args, patient_gt)

        # Evaluate objectives
        evaluation = evaluate_function(
            client, matched_results, diagnosis_name_for_evaluation
        )

        # Prepare output directory
        # Write evaluation results to JSON
        evaluation_dict = asdict(evaluation)

        evaluation_result = {
            "evaluation": evaluation_dict,
            "matched_results": jsonable_encoder(matched_results),
            "metadata": metadata,
        }

        evaluation_json = json.dumps(evaluation_result, indent=2)

        with open(output_file_path, "w") as output_file:
            output_file.write(evaluation_json)

    except Exception as e:
        logger.error(f"Error processing file {file_path}: {e}", exc_info=True)


MAP_PARAMS = {
    "UrineRequestList": "urine_values",
    "LabRequestList": "lab_values",
    "MicrobiologyRequestList": "microbiology_tests",
    "MedicationRequestList": "medications",
    "ProcedureSearch": "procedure",
    "ProcedureRequestFHIR": "procedure",
    "PatientHistory": None,
    "PhysicalExamination": None,
    "Finish": "diagnosis",
    "RadiologyRequestFHIR": ["modality", "region", "info"],
}


def extract_tool_use_human(med_assistant_outputs):
    """
    Extracts the tool use from the medical assistant outputs into structured format for comparison
    to the MIMIC IV ground truth.
    """
    tools_used = []
    for idx1, turn in enumerate(med_assistant_outputs):
        if turn["role"] == "tool_call":
            called = turn["tool_name"]
            param_field = MAP_PARAMS[called]
            if param_field is not None:
                if isinstance(param_field, list):
                    called_args = {pf: turn["parameters"].get(pf, None) for pf in param_field}
                else:
                    called_args = {param_field: turn["parameters"].get(param_field, None)}
            else:
                called_args = {}
            returned = turn["result"]
            flattened_called_args = _flatten_tool_args(called_args)
            tool_info = dict(
                called=called_args,
                called_args=flattened_called_args,
                returned=returned,
            )
            tools_used.append({turn["tool_name"]: tool_info})
    return tools_used


def _flatten_tool_args(called_args):
    """
    Flattens the tool arguments with edge case handling for different tools.
    """
    # handle blood values for now
    lab_val_args = called_args.get("lab_values", None)
    if lab_val_args is not None:
        requested_lab_values = []
        for lab_val_arg in lab_val_args:
            requested_lab_values.append(lab_val_arg["lab_value"])
        return requested_lab_values

    # handle urine values for now
    urine_val_args = called_args.get("urine_values", None)
    if urine_val_args is not None:
        requested_urine_values = []
        for urine_val_arg in urine_val_args:
            requested_urine_values.append(urine_val_arg["urine_value"])
        return requested_urine_values

    # handle microbiology tests
    microbiology_val_args = called_args.get("microbiology_tests", None)
    if microbiology_val_args is not None:
        requested_microbiology_tests = []
        for microbiology_val_arg in microbiology_val_args:
            requested_microbiology_tests.append(
                microbiology_val_arg["microbiology_value"]
            )
        return requested_microbiology_tests

    # handle radiology
    if "modality" in called_args.keys() and "region" in called_args.keys():
        called_args2 = copy.deepcopy(called_args)
        called_args2.pop("info", None)
        return called_args2

    # handle medications
    if "medications" in called_args.keys():
        return called_args["medications"]

    # base case
    else:
        return called_args


def merge_called_args(med_assistant_tools_used):
    """
    Merge the requests if tools were called multiple times,
    to facilitate evaluation.
    """
    called_args_dict = defaultdict(list)
    for item in med_assistant_tools_used:
        examination_name = list(item.keys())[0]
        called_args = list(item.values())[0]["called_args"]
        called_args_dict[examination_name].append(called_args)

    # if its a nested list, flatten it, ignore if its list of dicts or so
    for key, value in called_args_dict.items():
        if isinstance(value[0], list):
            called_args_dict[key] = sum(value, [])
    return called_args_dict


def evaluate_file_wrapper(
    client,
    file_path,
    dataset,
    out_dir,
    diagnosis,
    diagnosis_name_for_evaluation,
    evaluate_function,
):
    evaluate_one_file(
        client=client,
        file_path=file_path,
        dataset=dataset,
        out_dir=out_dir,
        diagnosis=diagnosis,
        diagnosis_name_for_evaluation=diagnosis_name_for_evaluation,
        evaluate_function=evaluate_function,
    )